In [12]:
import sys
print(sys.executable)

/Users/sahabtaali/Documents/Retention Analysis/commercial-retention-analysis/.venv/bin/python


In [13]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("../data/raw/Online Retail.xlsx")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(RAW_PATH)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [15]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [16]:
total_rows = len(df)
duplicate_rows = df.duplicated().sum()
missing_customer_pct = df["CustomerID"].isna().mean() * 100

df["line_revenue_gross"] = df["Quantity"] * df["UnitPrice"]

gross_revenue = df["line_revenue_gross"].sum()

df["is_cancelled_invoice"] = df["InvoiceNo"].astype(str).str.startswith("C")
df["is_refund_or_return"] = (df["Quantity"] < 0) | df["is_cancelled_invoice"]

cancelled_refund_revenue = df.loc[df["is_refund_or_return"], "line_revenue_gross"].sum()

df["is_positive_purchase"] = (
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0) &
    (~df["is_cancelled_invoice"])
)

analytical_purchase_revenue = df.loc[df["is_positive_purchase"], "line_revenue_gross"].sum()

df["is_identified_customer"] = df["CustomerID"].notna()

identifiable_customer_revenue = df.loc[
    df["is_positive_purchase"] & df["is_identified_customer"],
    "line_revenue_gross"
].sum()

valid_customers = df.loc[
    df["is_positive_purchase"] & df["is_identified_customer"],
    "CustomerID"
].nunique()

valid_orders = df.loc[
    df["is_positive_purchase"] & df["is_identified_customer"],
    "InvoiceNo"
].nunique()

audit_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "duplicate_rows",
        "missing_customer_pct",
        "gross_revenue",
        "cancelled_refund_revenue",
        "analytical_purchase_revenue",
        "identifiable_customer_revenue",
        "valid_customers",
        "valid_orders"
    ],
    "value": [
        total_rows,
        duplicate_rows,
        missing_customer_pct,
        gross_revenue,
        cancelled_refund_revenue,
        analytical_purchase_revenue,
        identifiable_customer_revenue,
        valid_customers,
        valid_orders
    ]
})

audit_summary

,metric,value
0,total_rows,5.419090e+05
1,duplicate_rows,5.268000e+03
2,missing_customer_pct,2.492669e+01
3,gross_revenue,9.747748e+06
4,cancelled_refund_revenue,-8.968125e+05
5,analytical_purchase_revenue,1.066668e+07
6,identifiable_customer_revenue,8.911408e+06
7,valid_customers,4.338000e+03
8,valid_orders,1.853200e+04


In [17]:
# Fix mixed-type columns before saving to parquet

clean_transactions["InvoiceNo"] = clean_transactions["InvoiceNo"].astype(str)
clean_transactions["StockCode"] = clean_transactions["StockCode"].astype(str)

clean_transactions["Description"] = clean_transactions["Description"].astype("string")
clean_transactions["Country"] = clean_transactions["Country"].astype("string")
clean_transactions["exclusion_reason"] = clean_transactions["exclusion_reason"].astype("string")

clean_transactions["CustomerID"] = clean_transactions["CustomerID"].astype("Int64")

clean_transactions.to_parquet(
    PROCESSED_DIR / "clean_transactions.parquet",
    index=False
)

audit_summary.to_csv(
    PROCESSED_DIR / "data_audit_summary.csv",
    index=False
)

In [18]:
test = pd.read_parquet(PROCESSED_DIR / "clean_transactions.parquet")
test.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,line_revenue_gross,is_cancelled_invoice,is_refund_or_return,is_positive_purchase,is_identified_customer,exclusion_reason,is_valid_order_line,analytical_revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,False,False,True,True,<NA>,True,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,False,False,True,True,<NA>,True,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,False,False,True,True,<NA>,True,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,False,False,True,True,<NA>,True,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,False,False,True,True,<NA>,True,20.34


In [19]:
clean_transactions.dtypes

InvoiceNo                            str
StockCode                            str
Description                       string
Quantity                           int64
InvoiceDate               datetime64[us]
UnitPrice                        float64
CustomerID                         Int64
Country                           string
line_revenue_gross               float64
is_cancelled_invoice                bool
is_refund_or_return                 bool
is_positive_purchase                bool
is_identified_customer              bool
exclusion_reason                  string
is_valid_order_line                 bool
analytical_revenue               float64
dtype: object